In [1]:
import wespeakerruntime as wespeaker

from torchaudio.datasets import LIBRISPEECH

import numpy as np
from os import listdir
import os
import pathlib

data_path = r'E:/code/SmartPeepHole/data/LibriSpeech/train-clean-100'

# lang='en' uses the VoxCeleb (English) model
# lang='chs' uses the CNCeleb (Chinese) model
spk = wespeaker.Speaker(lang="en")

ds = LIBRISPEECH(root="data", download=True)

embeddings = [[], []]

for audio_file in listdir(data_path + '/322/124146'):
    if audio_file.endswith('.flac'):
        embedding = spk.extract_embedding(os.path.join(data_path + '/322/124146', audio_file))[0, :]
        embeddings[0].append(embedding)

for audio_file in listdir(data_path + '/322/124147'):
    if audio_file.endswith('.flac'):
        embedding = spk.extract_embedding(os.path.join(data_path + '/322/124147', audio_file))[0, :]
        embeddings[1].append(embedding)

for embedding1, embedding2 in zip(embeddings[0], embeddings[1]):
    print(spk.compute_cosine_score(embedding1, embedding2))

e:\code\SmartPeepHole\.venv\lib\site-packages\torchaudio\_backend\utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


0.8548801
0.754472
0.893887
0.8976584
0.80707246
0.64864105
0.77015924
0.83430576
0.7860038
0.7681126
0.88127476
0.783169
0.81132364
0.8311856
0.77434784
0.7104371
0.8100494
0.8292567
0.6887788
0.74672437
0.64474356
0.8276628
0.7191366
0.789355
0.72158235
0.8697296
0.8046525
0.77642053
0.8815837
0.85079795
0.7936095
0.83957607
0.5843488
0.7638695
0.74364924
0.78611606
0.831118
0.67170095


In [3]:
local_path = r'E:/code/SmartPeepHole/data/Local'

embedding1 = spk.extract_embedding(os.path.join(local_path, "Rec1.wav"))[0, :]

embedding2 = spk.extract_embedding(os.path.join(local_path, "Rec2.wav"))[0, :]

print(spk.compute_cosine_score(embedding1, embedding2))

0.6451477


In [ ]:
from os import listdir
import os
from pydub import AudioSegment
import pathlib

decoders = []



for folder in listdir(data_path):
    if os.path.isdir(os.path.join(data_path, folder)):
        for chapter in listdir(os.path.join(data_path, folder)):
            if os.path.isdir(os.path.join(data_path, folder, chapter)):
                for file in listdir(os.path.join(data_path, folder, chapter)):
                    if file.endswith('.flac'):
                        flac_file_path = pathlib.PurePath(os.path.join(data_path, folder, chapter, file))
                        wav_file_path = pathlib.PurePath(os.path.join(r'./data/LibriSpeech/train-clean-100-wav/', folder, file)).with_suffix('.wav')
                        os.makedirs(wav_file_path.parent, exist_ok=True)
                        #print(f'Converting {flac_file_path} to {wav_file_path}')
                        try:
                            audio = AudioSegment.from_file(flac_file_path, format='flac')
                            audio.export(wav_file_path, format='wav')
                        except Exception as e:
                            print(f'Error converting {flac_file_path}: {e}')

In [4]:
from __future__ import annotations
import json
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import wespeakerruntime as wespeaker


def l2_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(x)
    return x / (n + eps)


class VoiceRegistry:
    """
    Simple on-disk registry:
      db/
        users.json   (metadata)
        alice.npy    (embedding template)
        bob.npy
    """
    def __init__(self, db_dir: str = "voice_db", lang: str = "en", onnx_path: Optional[str] = None):
        self.db_dir = Path(db_dir)
        self.db_dir.mkdir(parents=True, exist_ok=True)
        self.meta_path = self.db_dir / "users.json"

        if onnx_path:
            self.spk = wespeaker.Speaker(lang=lang, onnx_path=onnx_path)
        else:
            self.spk = wespeaker.Speaker(lang=lang)

        self._meta = self._load_meta()

    def _load_meta(self) -> Dict[str, dict]:
        if self.meta_path.exists():
            return json.loads(self.meta_path.read_text(encoding="utf-8"))
        return {}

    def _save_meta(self) -> None:
        self.meta_path.write_text(json.dumps(self._meta, indent=2), encoding="utf-8")

    def _user_path(self, user_id: str) -> Path:
        return self.db_dir / f"{user_id}.npy"

    def list_users(self) -> List[str]:
        return sorted(self._meta.keys())

    def register(self, user_id: str, wav_paths: List[str], resample_rate: int = 16000) -> np.ndarray:
        """
        Enroll a user from multiple utterances. Stores one template embedding.
        """
        if len(wav_paths) < 2:
            raise ValueError("Use at least 2 enrollment recordings (3-5 is better).")

        embs = []
        for p in wav_paths:
            e = self.spk.extract_embedding(p, resample_rate=resample_rate)
            embs.append(l2_normalize(np.asarray(e)))

        template = l2_normalize(np.mean(np.vstack(embs), axis=0))

        np.save(self._user_path(user_id), template.astype(np.float32))
        self._meta[user_id] = {"num_enroll_files": len(wav_paths)}
        self._save_meta()
        return template

    def load_template(self, user_id: str) -> np.ndarray:
        path = self._user_path(user_id)
        if not path.exists():
            raise KeyError(f"Unknown user: {user_id}")
        return np.load(path)

    def verify(self, user_id: str, wav_path: str, threshold: float = 0.65, resample_rate: int = 16000) -> Tuple[bool, float]:
        """
        1:1 verification (claimed identity).
        """
        template = l2_normalize(self.load_template(user_id))
        emb = l2_normalize(np.asarray(self.spk.extract_embedding(wav_path, resample_rate=resample_rate)))

        score = float(self.spk.compute_cosine_score(template, emb))
        return (score >= threshold), score

    def recognize(self, wav_path: str, threshold: float = 0.65, resample_rate: int = 16000) -> Tuple[Optional[str], float]:
        """
        1:N identification (who is it?). Returns (best_user or None, best_score).
        """
        users = self.list_users()
        if not users:
            return None, float("-inf")

        emb = l2_normalize(np.asarray(self.spk.extract_embedding(wav_path, resample_rate=resample_rate)))

        best_user = None
        best_score = float("-inf")
        for u in users:
            template = l2_normalize(self.load_template(u))
            s = float(self.spk.compute_cosine_score(template, emb))
            if s > best_score:
                best_score, best_user = s, u

        if best_score < threshold:
            return None, best_score
        return best_user, best_score
